In [ ]:
from scipy.ndimage import gaussian_filter1d

def harris_stephens_optimized(video, sigma=2, taph=1.5, s=2, k=0.005, theta=0.005):
    # Μετατροπή σε float32 για ακρίβεια και ταχύτητα
    video = video.astype(np.float32)
    
    # 1. Υπολογισμός Παραγώγων με Gaussian Derivatives
    # order=1 σημαίνει παράγωγος, order=0 σημαίνει απλό smoothing
    # axis=0:y, axis=1:x, axis=2:t (σύμφωνα με τη read_video σου)
    
    # Lx: παράγωγος ως προς x, smoothing στα y, t
    Lx = gaussian_filter1d(video, sigma, axis=1, order=1)
    Lx = gaussian_filter1d(Lx, sigma, axis=0, order=0)
    Lx = gaussian_filter1d(Lx, taph, axis=2, order=0)
    
    # Ly: παράγωγος ως προς y, smoothing στα x, t
    Ly = gaussian_filter1d(video, sigma, axis=0, order=1)
    Ly = gaussian_filter1d(Ly, sigma, axis=1, order=0)
    Ly = gaussian_filter1d(Ly, taph, axis=2, order=0)
    
    # Lt: παράγωγος ως προς t, smoothing στα x, y
    Lt = gaussian_filter1d(video, taph, axis=2, order=1)
    Lt = gaussian_filter1d(Lt, sigma, axis=0, order=0)
    Lt = gaussian_filter1d(Lt, sigma, axis=1, order=0)

    # 2. Υπολογισμός των στοιχείων του πίνακα M (Integration Scale)
    # Χρησιμοποιούμε μεγαλύτερο σ (integration scale = s * sigma)
    s = 2
    i_s, i_t = s * sigma, s * taph
    
    def integrate(term):
        res = gaussian_filter1d(term, i_s, axis=0)
        res = gaussian_filter1d(res, i_s, axis=1)
        return gaussian_filter1d(res, i_t, axis=2)

    M11, M22, M33 = integrate(Lx**2), integrate(Ly**2), integrate(Lt**2)
    M12, M13, M23 = integrate(Lx*Ly), integrate(Lx*Lt), integrate(Ly*Lt)

    # 3. Harris Criterion
    det = (M11 * (M22 * M33 - M23**2) - 
           M12 * (M12 * M33 - M13 * M23) + 
           M13 * (M12 * M23 - M13 * M22))
    trace = M11 + M22 + M33
    H = det - k * (trace**3)

    points = get_interest_points(H, sigma, theta=theta, N=500, method="Local-Maxima")
    return points

# Εισαγωγή Βιβλιοθηκών

In [25]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates
from scipy.ndimage import convolve1d
from scipy.ndimage import maximum_filter
import cv26_lab2_utils as p
import pickle
from sklearn.metrics import confusion_matrix
import copy

from torchvision.models.feature_extraction import create_feature_extractor
import torch
import torchvision as tv
import torch.nn.functional as F

# Μέρος 1: Παρακολούθηση Προσώπου και Χεριών με Χρήση της Μεθόδου Οπτικής Ροής των Lucas-Kanade

## 1.1 Παρακολούθηση Προσώπου και Χεριών

### 1.1.1

In [48]:
# Creating images paths with correct order 
image = [img for img in os.listdir("cv26_lab2_part1")]
image.sort(key=lambda img: int(img.split('.')[0]))
image_paths = [os.path.join("cv26_lab2_part1", img) for img in image]

# Initialize bounding boxes
initial_boxes = {
    "face": [154, 102, 67, 115],
    "left_hand": [93, 272, 56, 83],
    "right_hand": [201, 270, 56, 83]
}

In [49]:
# Gets points of interest located in In with coords xi and try to match them with 
# points in In-1 with coords xi+di. This function try to find where the points of interest
# in In comes from In-1

def lk(I1, I2, features, rho, epsilon, d_x0=None, d_y0=None):
    # I1: In-1 (Previous frame), I2: In (Current frame)
    # features: Points of interest found in In image
    
    # Number of features
    n_pts = features.shape[0] 
    
    # Get estimation if exists (eg. Multiscale Lucas-Kanade) else initialize to zero
    dx = d_x0.copy() if d_x0 is not None else np.zeros(n_pts, dtype=np.float32)  
    dy = d_y0.copy() if d_y0 is not None else np.zeros(n_pts, dtype=np.float32)

    # Pre-Calculate derivatives for In-1
    I1_x = cv2.Sobel(I1, cv2.CV_32F, 1, 0, ksize=3)
    I1_y = cv2.Sobel(I1, cv2.CV_32F, 0, 1, ksize=3)
    
    # Create Gaussian window. Defines the weighted neighborhoud of point
    half_w = int(np.ceil(3 * rho))
    win_size = int(2 * half_w + 1)
    G_1d = cv2.getGaussianKernel(win_size, rho)
    G_rho = (G_1d @ G_1d.T)
    
    # Relative coords for window (for example -5,5)
    rel_range = np.arange(-half_w, half_w + 1)
    rel_x, rel_y = np.meshgrid(rel_range, rel_range)

    for iteration in range(100): 
        new_dx = np.zeros_like(dx)
        new_dy = np.zeros_like(dy)
        
        for i in range(n_pts):
            xi, yi = features[i, 0], features[i, 1]
            
            # Window coords in In frame
            curr_x = xi + rel_x
            curr_y = yi + rel_y
            
            # Window coords in In-1 frame (window moved d)
            prev_x = curr_x + dx[i]
            prev_y = curr_y + dy[i]
            
            # Interpolation (get values of subpixels)
            # In(x)
            I2_val = map_coordinates(I2, [curr_y, curr_x], order=1)
            # In-1(x + di)
            I1_val = map_coordinates(I1, [prev_y, prev_x], order=1)
            # Derivatives In-1(x + di)
            I1x_val = map_coordinates(I1_x, [prev_y, prev_x], order=1)
            I2y_val = map_coordinates(I1_y, [prev_y, prev_x], order=1)
            
            # Calculate error E
            E = I2_val - I1_val
            
            # Weighted sums
            m11 = np.sum(G_rho * (I1x_val**2)) + epsilon
            m22 = np.sum(G_rho * (I2y_val**2)) + epsilon
            m12 = np.sum(G_rho * (I1x_val * I2y_val))
            
            b1 = np.sum(G_rho * (I1x_val * E))
            b2 = np.sum(G_rho * (I2y_val * E))
            
            # Solve with Cramer
            det = m11 * m22 - m12**2
            ux = (m22 * b1 - m12 * b2) / (det + 1e-9)
            uy = (-m12 * b1 + m11 * b2) / (det + 1e-9)
            
            # Update d
            dx[i] += ux
            dy[i] += uy
            
            # Keep correction values for early stop
            new_dx[i], new_dy[i] = ux, uy

        # Early stop criterion
        if np.max(np.abs(new_dx)) < 0.01 and np.max(np.abs(new_dy)) < 0.01:
            break
            
    return dx, dy

In [24]:
# Toy example: test LK on pure translations with varying rho and epsilon

def shift_image(I, tx, ty):
    rows, cols = I.shape[:2]
    M = np.float32([[1, 0, tx], [0, 1, ty]])
    return cv2.warpAffine(I, M, (cols, rows), borderMode=cv2.BORDER_REFLECT)

I1 = (cv2.imread(image_paths[0], cv2.IMREAD_GRAYSCALE) / 255.0).astype(np.float32)
x, y, w, h = map(int, initial_boxes["left_hand"])
crop1_full = I1[y:y+h, x:x+w]

# --- Experiment 1: varying rho (fixed epsilon=0.005) ---
print("=" * 60)
print("EXPERIMENT 1: varying rho  |  fixed epsilon=0.005")
print("=" * 60)
shifts = [(1,0), (2,0), (5,0), (8,0), (10,0), (15,0), (0,5), (5,5), (5,15)]
rho_values = [0.5, 1.0, 2.0, 3.0, 5.0, 10.0]
epsilon = 0.005
header = "{:<16} {:<6} {:<22} {:<10}".format("Shift (tx,ty)", "rho", "Est. (ux,uy)", "Error")
print(header)
print("-" * len(header))
prev_shift = None
for tx_real, ty_real in shifts:
    I2 = shift_image(I1, tx_real, ty_real)
    crop2 = I2[y:y+h, x:x+w]
    feats = cv2.goodFeaturesToTrack(np.float32(crop2), 30, 0.1, 5)
    if feats is None: continue
    feats = feats.reshape(-1, 2)
    if prev_shift != (tx_real, ty_real):
        print()
        prev_shift = (tx_real, ty_real)
    for rho in rho_values:
        dx, dy = lk(crop1_full, crop2, feats, rho=rho, epsilon=epsilon)
        ux = np.mean(-dx)
        uy = np.mean(-dy)
        err = np.sqrt((ux - tx_real)**2 + (uy - ty_real)**2)
        print(f"({tx_real:>3},{ty_real:>3})          {rho:<6} ({ux:>+6.2f}, {uy:>+6.2f})      {err:.4f}")

# --- Experiment 2: varying epsilon (fixed rho=3.0) ---
print()
print("=" * 60)
print("EXPERIMENT 2: varying epsilon  |  fixed rho=3.0")
print("=" * 60)
shifts_eps = [(1,0), (2,0), (5,0), (5,5)]
epsilon_values = [0.0001, 0.001, 0.005, 0.05, 0.1, 0.5, 1.0]
rho_fixed = 3.0
header2 = "{:<16} {:<9} {:<22} {:<10}".format("Shift (tx,ty)", "epsilon", "Est. (ux,uy)", "Error")
print(header2)
print("-" * len(header2))
prev_shift = None
for tx_real, ty_real in shifts_eps:
    I2 = shift_image(I1, tx_real, ty_real)
    crop2 = I2[y:y+h, x:x+w]
    feats = cv2.goodFeaturesToTrack(np.float32(crop2), 30, 0.1, 5)
    if feats is None: continue
    feats = feats.reshape(-1, 2)
    if prev_shift != (tx_real, ty_real):
        print()
        prev_shift = (tx_real, ty_real)
    for eps in epsilon_values:
        dx, dy = lk(crop1_full, crop2, feats, rho=rho_fixed, epsilon=eps)
        ux = np.mean(-dx)
        uy = np.mean(-dy)
        err = np.sqrt((ux - tx_real)**2 + (uy - ty_real)**2)
        print(f"({tx_real:>3},{ty_real:>3})          {eps:<9} ({ux:>+6.3f}, {uy:>+6.3f})      {err:.4f}")


EXPERIMENT 1: varying rho  |  fixed epsilon=0.005
Shift (tx,ty)    rho    Est. (ux,uy)           Error     
---------------------------------------------------------

(  1,  0)          0.5    ( +1.00,  +0.07)      0.0696
(  1,  0)          1.0    ( +0.94,  -0.01)      0.0606
(  1,  0)          2.0    ( +0.94,  -0.01)      0.0597
(  1,  0)          3.0    ( +0.94,  -0.01)      0.0587
(  1,  0)          5.0    ( +0.94,  -0.01)      0.0576
(  1,  0)          10.0   ( +0.94,  -0.01)      0.0582

(  2,  0)          0.5    ( +1.59,  -0.06)      0.4141
(  2,  0)          1.0    ( +1.86,  -0.29)      0.3192
(  2,  0)          2.0    ( +1.95,  -0.01)      0.0480
(  2,  0)          3.0    ( +1.94,  -0.02)      0.0625
(  2,  0)          5.0    ( +1.93,  -0.03)      0.0713
(  2,  0)          10.0   ( +1.93,  -0.01)      0.0692

(  5,  0)          0.5    ( +2.02,  -1.55)      3.3588
(  5,  0)          1.0    ( +1.92,  -1.67)      3.5055
(  5,  0)          2.0    ( +3.09,  -1.87)      2.6748
(  5, 

In [28]:
# This function takes pair of frames and calculate flow with costum lk and tvl1 algorithm, 
# visualize results and compare methods

def compare_lk_vs_tvl1(frame_pairs, lk_algorithm, output_dir, **params):

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    tvl1_obj = cv2.optflow.createOptFlow_DualTVL1()
    
    for i, (p1, p2) in enumerate(frame_pairs):
        # Load frames
        I_prev = (cv2.imread(p1, cv2.IMREAD_GRAYSCALE) / 255.0).astype(np.float32)
        I_curr = (cv2.imread(p2, cv2.IMREAD_GRAYSCALE) / 255.0).astype(np.float32)
        
        fig, ax = plt.subplots(figsize=(12, 10))
        ax.imshow(I_prev, cmap='gray')
        
        all_errors = []

        
            
        feats = cv2.goodFeaturesToTrack(np.float32(I_curr), 90, 0.1, 5)
        feats = feats.reshape(-1, 2)
            
        # Costum lk
        dx_lk, dy_lk = lk_algorithm(I_prev, I_curr, feats, **params)
        u_lk, v_lk = -dx_lk, -dy_lk
            
        # tvl1
        flow_tv_dense = tvl1_obj.calc(I_prev, I_curr, None)
            
        # Sample dense tvl1 flow at points of interest 
        coords = np.vstack((feats[:, 1] + dy_lk, feats[:, 0] + dx_lk)) # (y, x)
        u_tv = map_coordinates(flow_tv_dense[:, :, 0], coords, order=1, mode='nearest')
        v_tv = map_coordinates(flow_tv_dense[:, :, 1], coords, order=1, mode='nearest')
            
        # Calculate 'distance' between two methods
        dist = np.sqrt((u_lk - u_tv)**2 + (v_lk - v_tv)**2)
        avg_error = np.mean(dist)
        all_errors.append(avg_error)
            
        # Visualize
        # lk (red arrows)
        ax.quiver(feats[:, 0] + dx_lk, feats[:, 1] + dy_lk, u_lk, v_lk, 
                    color='red', angles='xy', scale=50, label='LK')
            
        # tvl1 (blue arrows)
        ax.quiver(feats[:, 0], feats[:, 1], u_tv, v_tv, 
                    color='cyan', angles='xy', scale=50, label='TV-L1')

            
        #rect = plt.Rectangle((x, y), w, h, fill=False, color='yellow', linestyle='--', alpha=0.5)
        #ax.add_patch(rect)
        #ax.text(x, y-5, f"{name} (AEPE: {avg_error:.3f})", color='yellow', fontsize=10)

        report = np.mean(all_errors)
        ax.set_title(f"Comparison: LK (Red) vs TV-L1 (Blue) | Frame Pair {i}\nDistance: {report:.4f}")
        ax.legend(loc='upper right')
        
        save_path = os.path.join(output_dir, f"comp_{i:03d}.png")
        plt.savefig(save_path)
        plt.close(fig)
        print(f"Saved comparison: {save_path} | Distance: {report:.4f}")

    print("Comparison Completed.")

In [29]:
# This cell runs comparison between custom lk and tvl1 at random frame pairs with different rho and epsilon values
# Results saved in lk-tvl1_comparison folder as images with both lk and tvl1 arrows painted

frame_pairs = [(image_paths[0],image_paths[1]), (image_paths[10],image_paths[11]),
               (image_paths[35],image_paths[36]), (image_paths[40],image_paths[41]),
               (image_paths[50],image_paths[51]), (image_paths[66],image_paths[67])]

rho_values   = [0.5, 1.0, 2.0, 5.0, 10.0]
epsilon_values = [0, 0.01, 0.05, 0.1, 1]

for rho in rho_values:
    for epsilon in epsilon_values:
        output_dir = f"./lk-tvl1_comparison/rho_{rho}_eps_{epsilon}"
        print(f"--- rho={rho}, epsilon={epsilon} ---")
        compare_lk_vs_tvl1(frame_pairs, lk, output_dir, rho=rho, epsilon=epsilon)


--- rho=0.5, epsilon=0 ---
Saved comparison: ./lk-tvl1_comparison/rho_0.5_eps_0\comp_000.png | Distance: 0.2282
Saved comparison: ./lk-tvl1_comparison/rho_0.5_eps_0\comp_001.png | Distance: 8.6952
Saved comparison: ./lk-tvl1_comparison/rho_0.5_eps_0\comp_002.png | Distance: 2.3832
Saved comparison: ./lk-tvl1_comparison/rho_0.5_eps_0\comp_003.png | Distance: 6.1058
Saved comparison: ./lk-tvl1_comparison/rho_0.5_eps_0\comp_004.png | Distance: 0.3406
Saved comparison: ./lk-tvl1_comparison/rho_0.5_eps_0\comp_005.png | Distance: 1.1944
Comparison Completed.
--- rho=0.5, epsilon=0.01 ---
Saved comparison: ./lk-tvl1_comparison/rho_0.5_eps_0.01\comp_000.png | Distance: 0.2286
Saved comparison: ./lk-tvl1_comparison/rho_0.5_eps_0.01\comp_001.png | Distance: 2.9041
Saved comparison: ./lk-tvl1_comparison/rho_0.5_eps_0.01\comp_002.png | Distance: 1.6562
Saved comparison: ./lk-tvl1_comparison/rho_0.5_eps_0.01\comp_003.png | Distance: 2.3734
Saved comparison: ./lk-tvl1_comparison/rho_0.5_eps_0.01\com

### 1.1.2

In [30]:
# Function calculating total displacement of box using threshold and median method
def displ(dx, dy, method='threshold', energy_threshold=0.005):
    
    if method == 'threshold':

        energy = dx**2 + dy**2
        mask = energy > energy_threshold
        
        if np.any(mask):
            displ_x = np.mean(dx[mask])
            displ_y = np.mean(dy[mask])
        else:
            displ_x, displ_y = 0.0, 0.0
            
    # Alternative solution moving box
    elif method == 'median':
        displ_x = np.median(dx)
        displ_y = np.median(dy)
        
    return np.array([displ_x, displ_y])

In [40]:
# Function using a flow calculation algorithm (lk, multiscale-lk, tvl1) and runs tracking of face, left and right hand of
# woman in dataset

def tracking(image_paths, method, initial_bboxes, output_dir, displ_method='threshold', energy_threshold=0.005, lk_algorithm=None, **params):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    bounding_boxes = copy.deepcopy(initial_bboxes) #deepcopy in order to copy both dict and 
    
    tvl1_obj = None
    if method == 'tvl1':
        tvl1_obj = cv2.optflow.createOptFlow_DualTVL1()
        
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Loop for frames pairs
    for t in range(len(image_paths)):
        # In-1 
        I_prev = (cv2.imread(image_paths[t], cv2.IMREAD_GRAYSCALE) / 255.0).astype(np.float32)
        # In 
        if t+1 > (len(image_paths)-1):
            I_curr = (cv2.imread(image_paths[len(image_paths)-1], cv2.IMREAD_GRAYSCALE) / 255.0).astype(np.float32)
        else:
            I_curr = (cv2.imread(image_paths[t+1], cv2.IMREAD_GRAYSCALE) / 255.0).astype(np.float32)
               
        ax.clear()
        ax.imshow(I_prev, cmap='gray')
        ax.set_title(f"Frame {t:03d} - Method: {method}")
        
        # Loop for head, left and right hand
        for name, box in bounding_boxes.items():
            x, y, w, h = map(int, box)
            # Create crops
            crop1 = I_prev[y:y+h, x:x+w]
            crop2 = I_curr[y:y+h, x:x+w]
            
            # Find features in crop2 
            feats = cv2.goodFeaturesToTrack(np.float32(crop2), 30, 0.1, 5)
            if feats is not None:
                feats = feats.reshape(-1, 2)
                
                
                if method in ['lk', 'multiscale']:
                    dx, dy = lk_algorithm(crop1, crop2, feats, **params)
                    
                elif method == 'tvl1':
                    flow = tvl1_obj.calc(crop1, crop2, None)
                    
                    coords = np.vstack((feats[:, 1], feats[:, 0]))
    
                    u = map_coordinates(flow[:, :, 0], coords, order=1, mode='nearest')
                    v = map_coordinates(flow[:, :, 1], coords, order=1, mode='nearest')
                    
                    dx = -u
                    dy = -v
                
                ux_box, uy_box = displ(dx, dy, method=displ_method, energy_threshold=energy_threshold)
                
                ax.quiver(feats[:, 0] + x + dx, feats[:, 1] + y + dy, -dx, -dy, 
                          color='r', angles='xy', scale=100)
    
                rect = plt.Rectangle((bounding_boxes[name][0], bounding_boxes[name][1]), w, h, 
                                     fill=False, color='g')
                ax.add_patch(rect)
                ax.text(bounding_boxes[name][0], bounding_boxes[name][1]-5, name, 
                        color='lime', fontsize=9)
                
                bounding_boxes[name][0] -= ux_box
                bounding_boxes[name][1] -= uy_box   
                
        save_path = os.path.join(output_dir, f"frame_{t:03d}.png")
        plt.savefig(save_path)
        print(f"Saved: {save_path}")
    
    plt.close(fig)
    print(f"Tracking completed for method: {method}")

In [147]:
# This cell runs experiment tracking with different parameters 
# 1.Change rho and epsilon of lk algorithm with displ_method='threshold' and threshold=0.005
# 2.Change method and threshold with rho=3.0 and epsilon=0.005
# Results in track_lk folder

#========================#
#1.Change rho and epsilon#
#========================#
rho_values = [0.5, 2.0, 3.0, 4.0, 10]
epsilon_values = [0, 0.01, 0.05, 0.1, 1]

for rho in rho_values:
    for epsilon in epsilon_values:
        print(f"--- Starting Simple LK Tracking with rho={rho} and epsilon={epsilon} ---")
        
        output_dir = f'./track_lk/rho_{rho}_eps_{epsilon}'
        
        tracking(
            image_paths=image_paths, 
            method='lk',
            initial_bboxes=initial_boxes, 
            output_dir=output_dir,
            lk_algorithm=lk,
            rho=rho,
            epsilon=epsilon
        )

#=============================#
#2.Change method and threshold#
#=============================#
energy_thresholds = [0.001, 0.01, 0.5, 1, 10]

for threshold in energy_thresholds:
    output_dir = f'./track_lk/threshold_{threshold}'
    
    tracking(
        image_paths=image_paths, 
        method='lk',
        initial_bboxes=initial_boxes, 
        output_dir=output_dir,
        displ_method='threshold',
        energy_threshold=threshold,
        lk_algorithm=lk,
        rho=3.0,
        epsilon=0.005
    )

output_dir = f'./track_lk/median'
    
tracking(
    image_paths=image_paths, 
    method='lk',
    initial_bboxes=initial_boxes, 
    output_dir=output_dir,
    displ_method='median',
    lk_algorithm=lk,
    rho=3.0,
    epsilon=0.005
)


print("All experiments finished.")

--- Starting Simple LK Tracking with rho=2.0 and epsilon=0.01 ---
Saved: ./track_lk/rho_2.0_eps_0.01\frame_000.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_001.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_002.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_003.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_004.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_005.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_006.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_007.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_008.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_009.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_010.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_011.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_012.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_013.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_014.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_015.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_016.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_017.png
Saved: ./track_lk/rho_2.0_eps_0.01\frame_018.png
Sav

In [ ]:
# Run tracking with tvl1
# Results in track_tvl1 folder

print("Starting TVL1 Tracking...")
tracking(
    image_paths=image_paths,
    method='tvl1', 
    initial_bboxes=initial_boxes,
    output_dir='./track_tvl1_lk',
    lk_algorithm=multi_scale_lk, 
    rho=3.0, 
    epsilon=0.05,
    n_scales=4                   
)

### 1.1.3

In [113]:
def multi_scale_lk(I1, I2, features, rho, epsilon, n_scales):

    sigma = 1

    pyramid1 = [I1]
    pyramid2 = [I2]
    
    filtered1 = I1
    filtered2 = I2
    
    for i in range(1, n_scales):

        kernel_size = int(np.ceil(3 * sigma) * 2 + 1)
    
        G_1d = cv2.getGaussianKernel(kernel_size, sigma)
        G_s = G_1d @ G_1d.T

        filtered1 = cv2.filter2D(filtered1, -1, G_s)
        filtered2 = cv2.filter2D(filtered2, -1, G_s)

        # Downsample and update filtered so each level is half the previous
        filtered1 = cv2.resize(filtered1, (0, 0), fx=0.5, fy=0.5)
        filtered2 = cv2.resize(filtered2, (0, 0), fx=0.5, fy=0.5)
        pyramid1.append(filtered1)
        pyramid2.append(filtered2)
    
    dx = np.zeros(len(features), dtype=np.float32)
    dy = np.zeros(len(features), dtype=np.float32)
     
    for s in reversed(range(n_scales)):
        curr_I1 = pyramid1[s]
        curr_I2 = pyramid2[s]
        
        curr_features = features / 2**s
        
        dx, dy = lk(curr_I1, curr_I2, curr_features, rho, epsilon, dx, dy)
        
        # 4. Μεταφορά στην επόμενη (μεγαλύτερη) κλίμακα
        if s > 0:
            dx *= 2.0
            dy *= 2.0
            
    return dx, dy

In [ ]:
# This cell runs tracking with custom multiscale-lk
# Results in track_multiscale_lk folder 

print("Starting Multiscale LK Tracking...")
tracking(
    image_paths=image_paths,
    method='multiscale',
    initial_bboxes=initial_boxes,
    output_dir='./track_multiscale_lk',
    lk_algorithm=multi_scale_lk,
    rho=3.0, 
    epsilon=0.05,
    n_scales=4                   
)

# Μέρος 2

## 2.1.0

In [33]:
#Function breaking 3D gaussian filter to 3 1D gaussian filters

def apply_3d_gaussian_filter(video, sigma_x, sigma_y, sigma_t):

    # Σ matrix is diagonal so we can break convolution with 3d gaussian filter to 3 convolutions with 1d gaussian filters
    kernel_x = cv2.getGaussianKernel(int(2 * np.ceil(3 * sigma_x) + 1), sigma_x).flatten()
    kernel_y = cv2.getGaussianKernel(int(2 * np.ceil(3 * sigma_y) + 1), sigma_y).flatten()
    if sigma_t:
        kernel_t = cv2.getGaussianKernel(int(2 * np.ceil(3 * sigma_t) + 1), sigma_t).flatten() 
    
    # Separate Convolutions with 1d gaussian filters
    L = convolve1d(input=video, weights=kernel_x, axis=1) # default mode reflect. Output size = Input size
    L = convolve1d(input=L, weights=kernel_y, axis=0)
    if sigma_t:
        L = convolve1d(input=L, weights=kernel_t, axis=2)
    return L

## 2.1.3 and 2.1.4

In [32]:
def get_interest_points(H, sigma, theta=0.005, N=500, method="Local-Maxima"):
    if method == "Local-Maxima":
        # Find max element in neighborhood
        local_max = maximum_filter(H, size=3)
       
        # Keep local maxima greater than threshold
        cond = (H == local_max) & (H > theta*np.max(H)) 
        y_coords, x_coords, t_coords = np.where(cond)
    
    if method == "Top-N":    
        flat_H = H.flatten()
        top_indices = np.argsort(flat_H)[-N:][::-1]
        y_coords, x_coords, t_coords = np.unravel_index(top_indices, H.shape)
    
    points = np.zeros((len(y_coords), 4))
    points[:, 0] = x_coords      # x 
    points[:, 1] = y_coords      # y 
    points[:, 2] = t_coords      # t 
    points[:, 3] = sigma         # sigma
    
    return points

## 2.1.1

In [31]:
def harris(video, sigma=4, taph=1.5, s=2, k=0.005, method='Local-Maxima', plot_H=False, save_path=None):
    # Smoothing
    L = apply_3d_gaussian_filter(video, sigma, sigma, taph)
    
    # Kernels for derivatives
    derivative_kernel = np.array([-1, 0, 1]) 
    
    # Compute derivatives
    Lx = convolve1d(input=L, weights=derivative_kernel, axis=1)
    Ly = convolve1d(input=L, weights=derivative_kernel, axis=0)
    Lt = convolve1d(input=L, weights=derivative_kernel, axis=2)
    
    i_s = s * sigma
    i_t = s * taph

    # Elements of J matrix
    M11 = apply_3d_gaussian_filter(Lx**2, i_s, i_s, i_t)
    M22 = apply_3d_gaussian_filter(Ly**2, i_s, i_s, i_t)
    M33 = apply_3d_gaussian_filter(Lt**2, i_s, i_s, i_t)
    M12 = apply_3d_gaussian_filter(Lx*Ly, i_s, i_s, i_t)
    M13 = apply_3d_gaussian_filter(Lx*Lt, i_s, i_s, i_t)
    M23 = apply_3d_gaussian_filter(Ly*Lt, i_s, i_s, i_t)
    
    # Harris-Stephens Criterion
    det = (M11 * (M22 * M33 - M23**2) - M12 * (M12 * M33 - M13 * M23) + M13 * (M12 * M23 - M13 * M22))   
    trace = M11 + M22 + M33
    H = det - k * (trace**3)

    points = get_interest_points(H, sigma, theta=0.001, N=600, method=method)
    print(f"Harris-Stephens found {points.shape[0]} points of interest\n")

    # If plot_H=True we save H as video with detected circles
    if (plot_H):
        H_filter = np.maximum(H, 0)
        p.show_detection(video_=H_filter, points_=points, save_path=save_path)
    
    return points

In [37]:
def multiscale_harris(video, sigma_0=4, taph_0=1.5, scale_step=np.sqrt(2), N=4, s_int=2, k=0.005, method='Local-Maxima'):
 
    sigma_vector = [sigma_0 * (scale_step**i) for i in range(N)]
    taph_vector = [taph_0 * (scale_step**i) for i in range(N)]
  
    LoG_table = []
    
    laplacian_kernel = np.array([1, -2, 1])
    
    for i in range(N):
        for j in range(N):
            sigma = sigma_vector[i]
            taph = taph_vector[j]
            
        
            L = apply_3d_gaussian_filter(video, sigma, sigma, taph)
            
            # 2. Υπολογισμός 2ων παραγώγων στους 3 άξονες
            Lxx = convolve1d(input=L, weights=laplacian_kernel, axis=1)
            Lyy = convolve1d(input=L, weights=laplacian_kernel, axis=0)
            Ltt = convolve1d(input=L, weights=laplacian_kernel, axis=2)
            
        
            LoG = np.abs((sigma**2) * (Lxx + Lyy) + (taph**2) * Ltt)
            LoG_table.append(LoG)
        
    final_points = []
    
    for i in range(N):
        for j in range(N):
            candidates = harris(
                video=video, 
                sigma=sigma_vector[i], 
                taph=taph_vector[j], 
                s=s_int, 
                k=k, 
                method=method, 
                plot_H=False
            )
            
            for pt in candidates:
                x, y, t = int(round(pt[0])), int(round(pt[1])), int(round(pt[2]))
                
                i_next = i+1
                j_next = j+1
                i_prev = i-1
                j_prev = j-1
                
                current_LoG = LoG_table[N*i+j][y, x, t]    
                next_LoG = LoG_table[N*i_next+j][y, x, t] if i_next < N else -1
                prev_LoG = LoG_table[N*i_prev+j][y, x, t] if i_prev > -1 else -1
                next_LoG1 = LoG_table[N*i+j_next][y, x, t] if j_next < N else -1
                prev_LoG1 = LoG_table[N*i+j_prev][y, x, t] if j_prev > -1 else -1
                
                if current_LoG > next_LoG and current_LoG > prev_LoG and current_LoG > next_LoG1 and current_LoG > prev_LoG1:
                    final_points.append([x, y, t, sigma_vector[i]])
                
    final_array = np.array(final_points)
    
    return final_array


## 2.1.2

In [34]:
def gabor_detector(video, sigma, taph, method='Local-Maxima', plot_H=False, save_path=None):

    # X, Y smoothing. Time axis remains as it was
    I_smoothed = apply_3d_gaussian_filter(video, sigma, sigma, 0)
    
    # Creating gabor filter for time axis
    t_limit = int(np.ceil(2 * taph))
    t = np.arange(-t_limit, t_limit + 1) # Time boundaries
    w = 4.0 / taph
    gaussian_part = np.exp(-t**2 / (2 * taph**2)) 
    h_ev = np.cos(2 * np.pi * t * w) * gaussian_part
    h_od = np.sin(2 * np.pi * t * w) * gaussian_part
    
    # L1 normalization (sum of abs values = 1)
    h_ev /= np.sum(np.abs(h_ev))
    h_od /= np.sum(np.abs(h_od))
    
    # Convolution with gabor filters
    res_ev = convolve1d(I_smoothed, h_ev, axis=2)
    res_od = convolve1d(I_smoothed, h_od, axis=2)
    
    # Criterion
    H = res_ev**2 + res_od**2

    points = get_interest_points(H, sigma, theta=0.05, N=600, method=method)
    print(f"Gabor-Detector found {points.shape[0]} points of interest\n")

    # If plot_H=True we save H as video with detected circles
    if (plot_H):
        p.show_detection(video_=H, points_=points, save_path=save_path)
    
    return points, H

In [20]:
def multiscale_gabor(video, sigma_0=4, taph_0=1.5, scale_step=np.sqrt(2), N=4, method='Local-Maxima'):
    
    sigma_vector = [sigma_0 * (scale_step**i) for i in range(N)]
    taph_vector = [taph_0 * (scale_step**i) for i in range(N)]

    LoG_table = []
    laplacian_kernel = np.array([1, -2, 1])

    for i in range(N):
        for j in range(N):
            sigma = sigma_vector[i]
            taph = taph_vector[j]

            L = apply_3d_gaussian_filter(video, sigma, sigma, taph)           
            
            Lxx = convolve1d(input=L, weights=laplacian_kernel, axis=1)
            Lyy = convolve1d(input=L, weights=laplacian_kernel, axis=0)
            Ltt = convolve1d(input=L, weights=laplacian_kernel, axis=2)      

            LoG = np.abs((sigma**2) * (Lxx + Lyy) + (taph**2) * Ltt)
            LoG_table.append(LoG)       

    final_points = []
    for i in range(N):
        for j in range(N):
            candidates = gabor_detector(video=video, sigma=sigma_vector[i], taph=taph_vector[j], method=method, plot_H=False)

            for pt in candidates:
                x, y, t = int(round(pt[0])), int(round(pt[1])), int(round(pt[2]))
         
                i_next = i+1
                j_next = j+1
                i_prev = i-1
                j_prev = j-1

    
                current_LoG = LoG_table[N*i+j][y, x, t]    
                next_LoG = LoG_table[N*i_next+j][y, x, t] if i_next < N else -1
                prev_LoG = LoG_table[N*i_prev+j][y, x, t] if i_prev > -1 else -1
                next_LoG1 = LoG_table[N*i+j_next][y, x, t] if j_next < N else -1
                prev_LoG1 = LoG_table[N*i+j_prev][y, x, t] if j_prev > -1 else -1              

                if current_LoG > next_LoG and current_LoG > prev_LoG and current_LoG > next_LoG1 and current_LoG > prev_LoG1:
                    final_points.append([x, y, t, sigma_vector[i]])
    
    final_array = np.array(final_points)  
    return final_array

## 2.1.4 Visualise Results of Implemented Algorithms

In [146]:
# Select videos from each category 
video_path_handclapping = "./cv26_lab2_part2_3/KTH/handclapping/person01_handclapping_d2_uncomp.avi" 
video_path_running = "./cv26_lab2_part2_3/KTH/running/person01_running_d1_uncomp.avi"
video_path_walking = "./cv26_lab2_part2_3/KTH/walking/person04_walking_d1_uncomp.avi"

# Load these videos
video_handclapping = p.read_video(video_path_handclapping) / 255.0
video_running = p.read_video(video_path_running) / 255.0
video_walking = p.read_video(video_path_walking) / 255.0

# Find points with detectors for every frame. plot_H=True means save H as video with detected circles
hs_points_handclapping = harris(video_handclapping, sigma=2, taph=1, s=1, k=0.005, plot_H=True, save_path="./video_detection/hs_handclapping_H")
mshs_points_handclapping =  multiscale_harris(video_handclapping, sigma_0=1.5, taph_0=0.5, scale_step=np.sqrt(2), N=4, s_int=1, k=0.005, method='Local-Maxima')
hs_points_handclapping_TopN = harris(video_handclapping, sigma=2, taph=1, s=1, k=0.005, method='Top-N')
hs_points_running = harris(video_running, sigma=2, taph=1, s=1, k=0.005, plot_H=True, save_path="./video_detection/hs_running_H")
hs_points_walking = harris(video_walking, sigma=2, taph=1, s=1, k=0.005, plot_H=True, save_path="./video_detection/hs_walking_H")

gb_points_handclapping = gabor_detector(video_handclapping,sigma=2, taph=1.5, plot_H=True, save_path="./video_detection/gb_handclapping_H")
msgb_points_handclapping =  multiscale_gabor(video_handclapping, sigma_0=2, taph_0=1.5, scale_step=np.sqrt(2), N=4, method='Local-Maxima')
gb_points_running = gabor_detector(video_running, sigma=2, taph=1.5, plot_H=True, save_path="./video_detection/gb_running_H")
gb_points_walking = gabor_detector(video_walking,sigma=2, taph=1.5, plot_H=True, save_path="./video_detection/gb_walking_H")

# Save results in folders
p.show_detection(video_=video_handclapping, points_=hs_points_handclapping, save_path="./video_detection/hs_handclapping")
p.show_detection(video_=video_handclapping, points_=mshs_points_handclapping, save_path="./video_detection/mshs_handclapping")
p.show_detection(video_=video_handclapping, points_=hs_points_handclapping_TopN, save_path="./video_detection/hs_handclapping_TopN")
p.show_detection(video_=video_running, points_=hs_points_running, save_path="./video_detection/hs_running")
p.show_detection(video_=video_walking, points_=hs_points_walking, save_path="./video_detection/hs_walking")

p.show_detection(video_=video_handclapping, points_=gb_points_handclapping, save_path="./video_detection/gb_handclapping")
p.show_detection(video_=video_handclapping, points_=msgb_points_handclapping, save_path="./video_detection/msgb_handclapping")
p.show_detection(video_=video_running, points_=gb_points_running, save_path="./video_detection/gb_running")
p.show_detection(video_=video_walking, points_=gb_points_walking, save_path="./video_detection/gb_walking")

Harris-Stephens found 376 points of interest

Harris-Stephens found 600 points of interest

Harris-Stephens found 145 points of interest

Harris-Stephens found 236 points of interest

Gabor-Detector found 676 points of interest

Gabor-Detector found 890 points of interest

Gabor-Detector found 1811 points of interest



## 2.2

In [25]:
# This function visualise derivative and flow fields. It is our Bonus thought and not main part of lab
def visualize_fields(ax, fig, img, ux, uy, title="Field", save_path=None):

    ax.clear() 
    ax.imshow(img, cmap='gray')
      
    step = 5
    y, x = np.mgrid[0:img.shape[0]:step, 0:img.shape[1]:step]
    u = ux[0:img.shape[0]:step, 0:img.shape[1]:step]
    v = uy[0:img.shape[0]:step, 0:img.shape[1]:step]
    
    plt.quiver(x, y, u, -v, color='red', scale=0.1, scale_units='xy', angles='xy')
    plt.title(title)
    
    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
    else:
        plt.show()

In [24]:
# Retunrs dict with 3 feature lists, one for each descriptor
def get_descriptors(video, points, nbins, grid_size, plot_fields=False):
    
    descriptors = {"HOG": [], "HOF": [], "HOG-HOF": []}
    
    unique_frames = np.unique(points[:, 2]).astype(int)

    oflow_method = cv2.optflow.DualTVL1OpticalFlow_create(nscales=1)

    if plot_fields:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    for f in unique_frames:
        # Current frame (uint8_t 0-255 and float64 0-1)
        img = video[:, :, f]
        img_u8 = (img * 255).astype(np.uint8)
        
        #---2.2.1---#
        # Derivative field
        kernel = np.array([-1, 0, 1])
        gx_full = convolve1d(img, kernel, axis=1)
        gy_full = convolve1d(img, kernel, axis=0)

        # Flow field
        prev_img_u8 = (video[:, :, f-1] * 255).astype(np.uint8) if f > 0 else img_u8
        flow_full = oflow_method.calc(prev_img_u8, img_u8, None)
        fx_full = flow_full[:, :, 0]
        fy_full = flow_full[:, :, 1]

        if plot_fields:
            visualize_fields(ax, fig, img, gx_full, gy_full, title="Derivative Field", save_path=f"./video_detection/vsl_der/frame{f}")        
            visualize_fields(ax, fig, img, fx_full, fy_full, title="Flow Field", save_path=f"./video_detection/vsl_flw/frame{f}")

        #---2.2.2---#
        # Get all points in same frame
        frame_points = points[points[:, 2] == f]
        for x, y, _, sigma in frame_points:
            patch_half = int(np.ceil(2 * sigma)) 
            
            # Patch limits
            y_min = int(max(y - patch_half, 0))
            y_max = int(min(y + patch_half, img.shape[0]))
            x_min = int(max(x - patch_half, 0))
            x_max = int(min(x + patch_half, img.shape[1]))
        
            # HOG descriptor 
            gx_patch = gx_full[y_min:y_max, x_min:x_max]
            gy_patch = gy_full[y_min:y_max, x_min:x_max]
            hog_desc = p.orientation_histogram(gx_patch, gy_patch, nbins, grid_size).flatten()

            # HOF descriptor 
            fx_patch = fx_full[y_min:y_max, x_min:x_max]
            fy_patch = fy_full[y_min:y_max, x_min:x_max]
            hof_desc = p.orientation_histogram(fx_patch, fy_patch, nbins, grid_size).flatten()

            # HOG-HOF descriptor
            full_desc = np.concatenate([hog_desc, hof_desc])
            
            descriptors["HOG"].append(hog_desc)
            descriptors["HOF"].append(hof_desc)
            descriptors["HOG-HOF"].append(full_desc)

    if plot_fields:
        plt.close(fig)
    
    return descriptors

## 2.3

### 2.3.1

In [35]:
# Feature extraction for 6 combinations. Returns dict results.
# results[detector_name][descriptor_name][_set] = list of feature_lists for all video of the _set
def FeatureExtraction():
    base_path = 'cv26_lab2_part2_3/KTH/' 
    train_split_file = 'cv26_lab2_part2_3/KTH/traininng_videos.txt' 
    categories = ['handclapping', 'running', 'walking']
    label_map = {'handclapping': 0, 'running': 1, 'walking': 2}
    
    detectors = {
        'Harris': lambda v: harris(v, sigma=2, taph=1, s=1, k=0.005),
        'Gabor-Detector': lambda v: gabor_detector(v, sigma=4, taph=1.5),
        'MultiScale-Harris': lambda v:   multiscale_harris(v, sigma_0=1.5, taph_0=0.5, scale_step=np.sqrt(2), N=4, s_int=1, k=0.005, method='Local-Maxima')
    }
    
    descriptor_types = ["HOG", "HOF", "HOG-HOF"]
    
    results = {}
    missing_combinations = []
    
    # This loop defines witch are the missing combinations to be calculated
    for det_name in detectors.keys():
        results[det_name] = {}
        for d_type in descriptor_types:
            filename = f"extracted_features/features_{det_name}_{d_type}.pkl"
            
            if os.path.exists(filename):
                print(f"Loading existing features: {filename}")
                with open(filename, 'rb') as f:
                    results[det_name][d_type] = pickle.load(f)
            else:
                print(f"Calculate features for combination: {det_name} - {d_type}")
                # Initialize empty struct for calculation
                results[det_name][d_type] = {"train_desc": [], "test_desc": [], "train_labels": [], "test_labels": []}
                missing_combinations.append((det_name, d_type))
    
    # If everything already exists exit
    if not missing_combinations:
        print("All features are already calculated")
    # Lets find features for missing combinations
    else:
        # Reading training videos names
        with open(train_split_file, 'r') as f:
            train_filenames = set(line.strip() for line in f if line.strip())
    
        print(f"\nStarting processing for {len(missing_combinations)} combinations...\n")
    
        for cat in categories:
            cat_path = os.path.join(base_path, cat)
            video_files = [f for f in os.listdir(cat_path) if f.endswith('.avi')]
            
            for v_file in video_files:
                print(f"Extracting features for video: {v_file}...\n")
                # Reading video
                video = p.read_video(os.path.join(cat_path, v_file)) / 255.0 
                is_train = v_file in train_filenames # Check if it is train video
                label = label_map[cat] # Putting labels for each video
                
                # Get only missing combinations
                needed_detectors = set(det for det, desc in missing_combinations)
                
                for det_name in needed_detectors:
                    # Find points of interest
                    points = detectors[det_name](video)
                    
                    # All descriptors for this detector
                    all_descs = get_descriptors(video, points, nbins=9, grid_size=np.array([4, 4]))
                    
                    for d_type in descriptor_types:         
                        target = results[det_name][d_type]
                        #---2.3.1---#
                        if is_train:
                            target["train_desc"].append(all_descs[d_type])
                            target["train_labels"].append(label)
                        else:
                            target["test_desc"].append(all_descs[d_type])
                            target["test_labels"].append(label)
    
        # Saving features only for missing combinations
        for det_name, d_type in missing_combinations:
            filename = f"extracted_features/features_{det_name}_{d_type}.pkl"
            with open(filename, 'wb') as f:
                pickle.dump(results[det_name][d_type], f)
            print(f"Saving file: {filename}")
    
    print("\nProcess completed")

    return results

### 2.3.2 and 2.3.3

In [40]:
# Main part for extracting features from dataset, create BoW, train SVM, and get results

D = 100  #Define number of visual words
detectors = ['Harris', 'Gabor-Detector', 'MultiScale-Harris']
descriptor_types = ["HOG", "HOF", "HOG-HOF"]

final_report = []

results = FeatureExtraction()

print(f"{'Detector':<20} | {'Descriptor':<10} | {'Accuracy':<10}")
print("-" * 50)

# Loop for 6 combinatios
for det_name in detectors:
    for desc_name in descriptor_types:

        data = results[det_name][desc_name]
            
        train_desc_raw = data["train_desc"] 
        test_desc_raw = data["test_desc"]
        
        train_desc = [np.array(v) for v in train_desc_raw]
        test_desc = [np.array(v) for v in test_desc_raw]

        train_labels = data["train_labels"]
        test_labels = data["test_labels"]

        #---2.3.2---#
        bow_train, bow_test = p.bag_of_words(train_desc, test_desc, num_centers=D)

        #---2.3.3---#
        accuracy, pred = p.svm_train_test(bow_train, train_labels, bow_test, test_labels)

        # Saving accuracy for each combination
        final_report.append({
            'detector': det_name,
            'descriptor': desc_name,
            'accuracy': accuracy
        })
        
        # Print results and confusion matrices
        print(f"{det_name:<20} | {desc_name:<10} | {accuracy:.2f}\n")
        print(confusion_matrix(test_labels, pred))
        print(50*"-")

#---2.3.4---#
best_case = max(final_report, key=lambda x: x['accuracy'])
print("-" * 50)
print(f"Best Combination: {best_case['detector']} + {best_case['descriptor']} with {best_case['accuracy']:.2f}% accuracy")

Loading existing features: extracted_features/features_Harris_HOG.pkl
Loading existing features: extracted_features/features_Harris_HOF.pkl
Loading existing features: extracted_features/features_Harris_HOG-HOF.pkl
Loading existing features: extracted_features/features_Gabor-Detector_HOG.pkl
Loading existing features: extracted_features/features_Gabor-Detector_HOF.pkl
Loading existing features: extracted_features/features_Gabor-Detector_HOG-HOF.pkl
Loading existing features: extracted_features/features_MultiScale-Harris_HOG.pkl
Loading existing features: extracted_features/features_MultiScale-Harris_HOF.pkl
Loading existing features: extracted_features/features_MultiScale-Harris_HOG-HOF.pkl
All features are already calculated

Process completed
Detector             | Descriptor | Accuracy  
--------------------------------------------------
Harris               | HOG        | 0.92

[[4 0 0]
 [0 3 1]
 [0 0 4]]
--------------------------------------------------
Harris               | HOF 

## Bonus

### Visualize derivative and flow fields used by HOG, HOF detectors

In [12]:
video_path = "./cv26_lab2_part2_3/KTH/handclapping/person01_handclapping_d2_uncomp.avi" 
video = p.read_video(video_path) / 255.0
points = harris_stephens(video, sigma=2, taph=1, s=1, k=0.005)
all_descs = get_descriptors(video, points, nbins=9, grid_size=np.array([4, 4]), plot_fields=True)

Harris-Stephens found 376 points of interest



### Create GIFs for every visualized result

In [50]:
#-----Create GIFs------#
import imageio
import re

def extract_number(filename):
    match = re.search(r'\d+', filename)
    return int(match.group()) if match else 0

base_path = './video_detection' 

for fld in os.listdir(base_path):
    fld_path = os.path.join(base_path, fld)
                   
    images_list = [img for img in os.listdir(fld_path)] 
    images_list.sort(key=extract_number)

    frames = []
    for file_name in images_list:
        file_path = os.path.join(fld_path, file_name)
        frames.append(imageio.v2.imread(file_path)) 
            
    output_gif_path = os.path.join('./GIFs', fld + ".gif")
    imageio.mimsave(output_gif_path, frames, fps=60)
    print(f"Το GIF {fld}.gif δημιουργήθηκε.")

print("Process Completed")

Το GIF gb_handclapping.gif δημιουργήθηκε.
Το GIF gb_handclapping_H.gif δημιουργήθηκε.
Το GIF gb_running.gif δημιουργήθηκε.
Το GIF gb_running_H.gif δημιουργήθηκε.
Το GIF gb_walking.gif δημιουργήθηκε.
Το GIF gb_walking_H.gif δημιουργήθηκε.
Το GIF hs_handclapping.gif δημιουργήθηκε.
Το GIF hs_handclapping_H.gif δημιουργήθηκε.
Το GIF hs_handclapping_TopN.gif δημιουργήθηκε.
Το GIF hs_running.gif δημιουργήθηκε.
Το GIF hs_running_H.gif δημιουργήθηκε.
Το GIF hs_walking.gif δημιουργήθηκε.
Το GIF hs_walking_H.gif δημιουργήθηκε.
Το GIF msgb_handclapping.gif δημιουργήθηκε.
Το GIF mshs_handclapping.gif δημιουργήθηκε.
Το GIF vsl_der.gif δημιουργήθηκε.
Το GIF vsl_flw.gif δημιουργήθηκε.
Process Completed


# Μέρος 3

## 3.1

In [41]:
#Importing CNN model pretrained
weights=tv.models.MobileNet_V3_Small_Weights.DEFAULT
model_2d = tv.models.mobilenet_v3_small(weights=weights)

#Necesssary Transforms/Preprocess
preprocessor = weights.transforms()

#Removing classifier
layer_nodes={'avgpool': 'cnn_descriptor'}
feature_extractor_2d = create_feature_extractor(model=model_2d, return_nodes=layer_nodes)

In [56]:
def FeatureExtractionCNN2D(feature_extractor, preprocessor, loadfile=None, savefile=None):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    feature_extractor.to(device)
    #Change extractor to evaluation mode
    feature_extractor.eval()

    #Loading existing features from previous run
    if loadfile is not None and os.path.exists(loadfile):
        print(f"Loading data from {loadfile}...")
        with open(loadfile, 'rb') as f:
            saved_data = pickle.load(f)
        print(f"Data loaded from {loadfile}")
        return saved_data 

    # Struct for saving extracted features
    features = {"train_desc" : [], "train_labels" : [],  "test_desc" : [], "test_labels" : []}
    
    train_split_file = "./cv26_lab2_part2_3/KTH/trainning_videos.txt"
    with open(train_split_file, 'r') as f:
        train_filenames = set(line.strip() for line in f if line.strip())
    
    dataDir = './cv26_lab2_part2_3/UCF11/'
    categories = ['basketball', 'biking', 'diving', 'swing', 'volleyball_spiking', 'walking']
            

    with torch.no_grad():
        # Reading dataset
        for class_idx, class_name in enumerate(categories):
            folder_path = os.path.join(dataDir, class_name)
            
            for filename_idx, filename in enumerate(os.listdir(folder_path)):
                temp_features_list = []
                is_train = filename in train_filenames
                v_path = os.path.join(folder_path, filename)
                #---3.1.1---#
                video = (p.read_video(v_path, gray=False) / 255.0).astype(np.float32)

                #---3.1.2---#
                # Iteration for each frame of same video
                for f in range(video.shape[2]):
                    frame = video[:, :, f, :].astype(np.float32) #cv2 needs float32
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    
                    frame_tensor = tv.transforms.functional.to_tensor(frame)
                    tensor_clean = preprocessor(frame_tensor).unsqueeze(0).to(device) #Suitable for CNN
                    out_clean = feature_extractor(tensor_clean) #Extraction of features
                    temp_features_list.append(out_clean['cnn_descriptor'].squeeze().cpu().numpy()) #Saving features in a list

                # Separate train and test set
                if is_train:
                    features["train_desc"].append(np.mean(temp_features_list, axis=0)) # Average pooling
                    features["train_labels"].append(class_idx)
                else:
                    features["test_desc"].append(np.mean(temp_features_list, axis=0)) # Average pooling
                    features["test_labels"].append(class_idx)
                

    #Saving features to file
    if savefile is not None:
        with open(savefile, 'wb') as f:
            pickle.dump(features, f)
            
    return features

In [43]:
#---3.1.3---#
features_2d = FeatureExtractionCNN2D(feature_extractor_2d, preprocessor, loadfile="./extracted_features/featuresCNN_2d.pkl", savefile=None)
train_desc_cnn2D = np.array(features_2d["train_desc"])
test_desc_cnn2D = np.array(features_2d["test_desc"])

accuracy_2d, pred_2d = p.svm_train_test(train_desc_cnn2D, features_2d["train_labels"], test_desc_cnn2D, features_2d["test_labels"], svm_type='linear')

Loading data from ./extracted_features/featuresCNN_2d.pkl...
Data loaded from ./extracted_features/featuresCNN_2d.pkl


## 3.2

In [44]:
#---3.2.1---#
model_3d = torch.hub.load('facebookresearch/pytorchvideo', 'x3d_xs', pretrained=True)

return_nodes = {'blocks.5.output_pool': 'features_3d'}
feature_extractor_3d = create_feature_extractor(model_3d, return_nodes=return_nodes)

Using cache found in C:\Users\feida/.cache\torch\hub\facebookresearch_pytorchvideo_main


In [54]:
def FeatureExtractionCNN3D(feature_extractor, loadfile=None, savefile=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    #Change extractor to evaluation mode
    feature_extractor.to(device).eval()

    mean = torch.tensor([0.45, 0.45, 0.45]).view(1, 3, 1, 1, 1).to(device)
    std = torch.tensor([0.225, 0.225, 0.225]).view(1, 3, 1, 1, 1).to(device)

    feature_extractor.to(device)

    #Loading existing features from previous run
    if loadfile is not None and os.path.exists(loadfile):
        print(f"Loading data from {loadfile}...")
        with open(loadfile, 'rb') as f:
            saved_data = pickle.load(f)
        print(f"Data loaded from {loadfile}")
        return saved_data 

    features = {"train_desc" : [], "train_labels" : [],  "test_desc" : [], "test_labels" : []}
    
    train_split_file = "./cv26_lab2_part2_3/UCF11/training_videos.txt"
    with open(train_split_file, 'r') as f:
        train_filenames = set(line.strip() for line in f if line.strip())
    
    dataDir = './cv26_lab2_part2_3/UCF11/'
    categories = ['basketball', 'biking', 'diving', 'swing', 'volleyball_spiking', 'walking']
            

    #---3.2.2---#
    with torch.no_grad():
        # Reading dataset
        for class_idx, class_name in enumerate(categories):
            folder_path = os.path.join(dataDir, class_name)
            
            for filename_idx, filename in enumerate(os.listdir(folder_path)):
                is_train = filename in train_filenames
                v_path = os.path.join(folder_path, filename)
                video = p.read_video(v_path, gray=False)
                
                # (H, W, T, C) -> (C, T, H, W)
                video_tensor = torch.from_numpy(video).permute(3, 2, 0, 1).float() / 255.0
                
                # Add batch dimension -> (1, C, T, H, W)
                video_tensor = video_tensor.unsqueeze(0).to(device)
                 
                # Interpolation: T=4, H=182, W=182
                video_volume = F.interpolate(video_tensor, size=(4, 182, 182), mode='trilinear', align_corners=False)
                
                # Normalization
                video_volume = (video_volume - mean) / std
                
                # Feature Extraction
                output = feature_extractor(video_volume)
                # Flattening
                temp_features = output['features_3d'].flatten().cpu().numpy()
                
                if is_train:
                    features["train_desc"].append(temp_features)
                    features["train_labels"].append(class_idx)
                else:
                    features["test_desc"].append(temp_features)
                    features["test_labels"].append(class_idx)
                    
    #Saving features to file
    if savefile is not None:
        with open(savefile, 'wb') as f:
            pickle.dump(features, f) 
            
    return features

In [55]:
def FeatureExtractionCNN3D_alt(feature_extractor, loadfile=None, savefile=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    #Change extractor to evaluation mode
    feature_extractor.to(device).eval()

    mean = torch.tensor([0.45, 0.45, 0.45]).view(1, 3, 1, 1, 1).to(device)
    std = torch.tensor([0.225, 0.225, 0.225]).view(1, 3, 1, 1, 1).to(device)

    feature_extractor.to(device)

    #Loading existing features from previous run
    if loadfile is not None and os.path.exists(loadfile):
        print(f"Loading data from {loadfile}...")
        with open(loadfile, 'rb') as f:
            saved_data = pickle.load(f)
        print(f"Data loaded from {loadfile}")
        return saved_data 

    features = {"train_desc" : [], "train_labels" : [],  "test_desc" : [], "test_labels" : []}
    
    train_split_file = "./cv26_lab2_part2_3/UCF11/training_videos.txt"
    with open(train_split_file, 'r') as f:
        train_filenames = set(line.strip() for line in f if line.strip())
    
    dataDir = './cv26_lab2_part2_3/UCF11/'
    categories = ['basketball', 'biking', 'diving', 'swing', 'volleyball_spiking', 'walking']
            

    #---3.2.2---#
    with torch.no_grad():
        # Reading dataset
        for class_idx, class_name in enumerate(categories):
            folder_path = os.path.join(dataDir, class_name)
            
            for filename_idx, filename in enumerate(os.listdir(folder_path)):
                is_train = filename in train_filenames
                v_path = os.path.join(folder_path, filename)
                video = p.read_video(v_path, gray=False)
                
                # (H, W, T, C) -> (C, T, H, W)
                video_tensor = torch.from_numpy(video).permute(3, 2, 0, 1).float() / 255.0

                total_frames = video_tensor.shape[1]
                clip_features = []
                
                for i in range(0, total_frames - 3, 4):
                    clip = video_tensor[:, i:i+4, :, :] # Επιλογή 4 frames
                    clip = clip.unsqueeze(0).to(device) # (1, C, 4, H, W)
                    
                    # Χωρική προσαρμογή μόνο (H=182, W=182) - Ο χρόνος είναι ήδη T=4
                    clip = F.interpolate(clip, size=(4, 182, 182), mode='trilinear', align_corners=False)
                    clip = (clip - mean) / std
                    
                    output = feature_extractor(clip)
                    clip_features.append(output['features_3d'].flatten().cpu().numpy())
                
                if clip_features:
                    # Average pooling όλων των clips του συγκεκριμένου βίντεο
                    video_descriptor = np.mean(clip_features, axis=0)
                    
                    if is_train:
                        features["train_desc"].append(video_descriptor)
                        features["train_labels"].append(class_idx)
                    else:
                        features["test_desc"].append(video_descriptor)
                        features["test_labels"].append(class_idx)
                        
    if savefile is not None:
        with open(savefile, 'wb') as f:
            pickle.dump(features, f) 
            
    return features

In [57]:
#---3.2.3---#
features_3d = FeatureExtractionCNN3D(feature_extractor_3d, loadfile="./extracted_features/featuresCNN_3d.pkl", savefile=None)
train_desc_cnn3D = np.array(features_3d["train_desc"])
test_desc_cnn3D = np.array(features_3d["test_desc"])

features_3d_alt = FeatureExtractionCNN3D_alt(feature_extractor_3d, savefile="./extracted_features/featuresCNN_3d_alt.pkl")
train_desc_cnn3D_alt = np.array(features_3d_alt["train_desc"])
test_desc_cnn3D_alt = np.array(features_3d_alt["test_desc"])

accuracy_3d, pred_3d = p.svm_train_test(train_desc_cnn3D, features_3d["train_labels"], test_desc_cnn3D, features_3d["test_labels"], svm_type='linear')
accuracy_3d_alt, pred_3d_alt = p.svm_train_test(train_desc_cnn3D_alt, features_3d_alt["train_labels"], test_desc_cnn3D_alt, features_3d_alt["test_labels"], svm_type='linear')

Loading data from ./extracted_features/featuresCNN_3d.pkl...
Data loaded from ./extracted_features/featuresCNN_3d.pkl


In [58]:
#---3.2.4---#
print(f"Results from 2D CNN\n")
print(f"Accuracy: {accuracy_2d:.3f}\n")
print(confusion_matrix(features_2d["test_labels"], pred_2d))

print()
print(21*'-')

print(f"\nResults from 3D CNN\n")
print(f"Accuracy: {accuracy_3d:.3f}\n")
print(confusion_matrix(features_3d["test_labels"], pred_3d))

print()
print(21*'-')

print(f"\nResults from 3D CNN (alternative solution) \n")
print(f"Accuracy: {accuracy_3d_alt:.3f}\n")
print(confusion_matrix(features_3d_alt["test_labels"], pred_3d_alt))

Results from 2D CNN

Accuracy: 0.988

[[28  0  0  0  0  1]
 [ 1 28  0  0  0  0]
 [ 0  0 32  0  0  0]
 [ 0  0  0 28  0  0]
 [ 0  0  0  0 24  0]
 [ 0  0  0  0  0 25]]

---------------------

Results from 3D CNN

Accuracy: 0.964

[[28  0  0  1  0  0]
 [ 0 29  0  0  0  0]
 [ 0  0 32  0  0  0]
 [ 0  0  0 28  0  0]
 [ 1  0  0  0 23  0]
 [ 0  2  0  2  0 21]]

---------------------

Results from 3D CNN (alternative solution) 

Accuracy: 0.988

[[28  0  0  0  1  0]
 [ 0 29  0  0  0  0]
 [ 0  0 32  0  0  0]
 [ 0  0  0 28  0  0]
 [ 0  0  0  0 24  0]
 [ 0  1  0  0  0 24]]
